In [27]:
#TASK 1
# %pip install pandas
import pandas as pd

df = pd.read_csv('cleaned_ebay_deals.csv')

#even tho already done in cleaning
df['price'] = pd.to_numeric(df['price'].replace('[\$,]', '', regex=True), errors='coerce')
df['original_price'] = pd.to_numeric(df['original_price'].replace('[\$,]', '', regex=True), errors='coerce')
df['shipping_numeric'] = df['shipping'].apply(lambda x: 0 if 'free' in str(x).lower() else x)
df['shipping_numeric'] = pd.to_numeric(df['shipping_numeric'], errors='coerce').fillna(0)

print("Dataset Preview:")
display(df.head())
print(f"\nDataset Shape: {df.shape}")
print("\nMissing Values Per Column:")
print(df.isnull().sum())
#note: .head() with empty () prints 5 rows by default

# print("\nFinal Data Types:")
# print(df.dtypes)

Dataset Preview:


<>:8: SyntaxWarning: invalid escape sequence '\$'
<>:9: SyntaxWarning: invalid escape sequence '\$'
<>:8: SyntaxWarning: invalid escape sequence '\$'
<>:9: SyntaxWarning: invalid escape sequence '\$'
C:\Users\hp\AppData\Local\Temp\ipykernel_7844\126181748.py:8: SyntaxWarning: invalid escape sequence '\$'
  df['price'] = pd.to_numeric(df['price'].replace('[\$,]', '', regex=True), errors='coerce')
C:\Users\hp\AppData\Local\Temp\ipykernel_7844\126181748.py:9: SyntaxWarning: invalid escape sequence '\$'
  df['original_price'] = pd.to_numeric(df['original_price'].replace('[\$,]', '', regex=True), errors='coerce')


,timestamp,title,price,original_price,shipping,item_url,discount_amount,discount_percentage,shipping_numeric
0,2026-03-10 19:23:28,Apple iPhone 15 Pro 256GB Unlocked AT&T T-Mobi...,523.95,1099.00,Free shipping,https://www.ebay.com/itm/305315521632?_trkparm...,575.05,52.32,0.0
1,2026-03-10 19:23:28,Samsung Galaxy S23 128GB S911U Unlocked - Exce...,239.99,799.99,Shipping info unavailable,https://www.ebay.com/itm/256372271129?_trkparm...,560.00,70.00,0.0
2,2026-03-10 19:23:29,Samsung Galaxy S21 5G 128GB G991U Unlocked - Good,134.99,799.99,Shipping info unavailable,https://www.ebay.com/itm/254987231741?_trkparm...,665.00,83.13,0.0
3,2026-03-10 19:23:29,Samsung Galaxy S23 Ultra S918U1 256GB Unlocked...,402.99,999.99,Shipping info unavailable,https://www.ebay.com/itm/205816814214?_trkparm...,597.00,59.70,0.0
4,2026-03-10 19:23:29,Microsoft Xbox Wireless Controller For Xbox Se...,45.95,80.00,Shipping info unavailable,https://www.ebay.com/itm/275287871839?_trkparm...,34.05,42.56,0.0



Dataset Shape: (263, 9)

Missing Values Per Column:
timestamp              0
title                  0
price                  0
original_price         0
shipping               0
item_url               0
discount_amount        0
discount_percentage    0
shipping_numeric       0
dtype: int64


In [28]:
# TASK 2: PART A: PRICE-BASED
df['effective_price']=df['price']+df['shipping_numeric']
df['discount_ratio']=((df['original_price']-df['price'])/df['original_price']).fillna(0)
# #fillna(0): if original price is missing or ratio NaN its 0
# display(df[['title', 'price', 'original_price','effective_price', 'discount_ratio']].head(50))
#note: WHY ARE WE DOING SHIPPING NUMERIC IF THE DATA WE SCRAPED MA FIYA NUMERIC SHIPPING???

In [29]:
#Task 2: Part B: Title
df['title_len']=df['title'].str.len()
df['title_word_count']=df['title'].apply(lambda x: len(str(x).split()))

key_words=['new', 'used', 'refurbished', 'bundle', 'case', 'charger']
for word in key_words:
    df[f'has_{word}']=df['title'].str.contains(word, case=False).astype(int) #case false for case semsitivity
    
# print("Part B: Title Features")
# display(df[['title', 'has_new', 'has_refurbished', 'has_bundle']].head(50))

In [30]:
# task 2: part c time
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.weekday

# print("Part C: Time-Based Features")
# display(df[['timestamp', 'hour', 'weekday']].head())

df.to_csv('ebay_features.csv', index=False)

In [31]:
# %pip install --no-cache-dir --user scikit-learn

In [32]:
#Task 3
# %pip install scikit-learn
from sklearn.model_selection import train_test_split

df['high_discount'] = (df['discount_percentage'] > 20).astype(int)

features = [
    'price', 'original_price', 'shipping_numeric', 'discount_ratio',
    'title_len', 'title_word_count', 'hour', 'weekday',
    'has_new', 'has_used', 'has_refurbished', 'has_bundle', 'has_case', 'has_charger'
]

X = df[features]
y = df['high_discount']

#prevent overfitting=>dont show it all the data
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, #test 20%, train 80%
    random_state=42, 
    stratify=y
)


In [33]:
#TASK 4:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score,confusion_matrix

#so that we can save in txt automatically and not manually
results_text=[]
def evaluate_model(name, y_true, y_pred):
    output = f"\n{name} Results:\n"
    output += f"Accuracy: {accuracy_score(y_true, y_pred):.4f}\n"
    output += f"Precision: {precision_score(y_true, y_pred):.4f}\n"
    output += f"Recall: {recall_score(y_true, y_pred):.4f}\n"
    output += f"F1 Score: {f1_score(y_true, y_pred):.4f}\n"
    output += f"Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n"
    output += "-"*30 + "\n"
    
    print(output)
    results_text.append(output)


In [34]:
# model 1: only price and original price
baseline_cols=['price','original_price']
model_baseline=LogisticRegression()
model_baseline.fit(X_train[baseline_cols],y_train)
y_pred_baseline=model_baseline.predict(X_test[baseline_cols])
evaluate_model("Logistic Regression (Baseline)", y_test, y_pred_baseline)


Logistic Regression (Baseline) Results:
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000
Confusion Matrix:
[[16  0]
 [ 0 37]]
------------------------------



In [35]:
# model2: logistics regression, feature engineering for extended featues
model_extended=LogisticRegression(max_iter=1000) #make it think longer
model_extended.fit(X_train,y_train)
y_pred_extended=model_extended.predict(X_test)
evaluate_model("Logistic Regression(extended):", y_test, y_pred_extended)


Logistic Regression(extended): Results:
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000
Confusion Matrix:
[[16  0]
 [ 0 37]]
------------------------------



In [36]:
#model3: decision tree classigier
model_tree=DecisionTreeClassifier(max_depth=5, random_state=42)
model_tree.fit(X_train, y_train)
y_pred_tree=model_tree.predict(X_test)
evaluate_model("Decision Tree Classifier", y_test, y_pred_tree)


Decision Tree Classifier Results:
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000
Confusion Matrix:
[[16  0]
 [ 0 37]]
------------------------------



In [37]:
with open("model_results.txt", "w") as f:
    f.writelines(results_text)

print("File 'model_results.txt' has been created successfully!")

File 'model_results.txt' has been created successfully!


In [38]:
print("Current balance of the data:")
print(df['high_discount'].value_counts(normalize=True))

minority = df[df['high_discount'] == 0] #normal deals
majority = df[df['high_discount'] == 1] # HOT DEALS🔥🔥🔥🔥🔥🧨🧨🧨🧨🧨🧨
#take a random sample of normal deals that is the same size as hot deals
balanced_df = pd.concat([majority.sample(len(minority), random_state=42), minority])
print(f"New balanced dataset size: {len(balanced_df)}")
print("NEW BALANCE CHECK:")
print(balanced_df['high_discount'].value_counts(normalize=True))

Current balance of the data:
high_discount
1    0.695817
0    0.304183
Name: proportion, dtype: float64
New balanced dataset size: 160
NEW BALANCE CHECK:
high_discount
1    0.5
0    0.5
Name: proportion, dtype: float64


In [39]:
X_bal = balanced_df[features]
y_bal = balanced_df['high_discount']

X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

#retrain using decision tree
model_balanced = DecisionTreeClassifier(max_depth=5, random_state=42)
model_balanced.fit(X_train_bal, y_train_bal)

y_pred_bal = model_balanced.predict(X_test_bal)

In [40]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

acc_bal = accuracy_score(y_test_bal, y_pred_bal)
prec_bal = precision_score(y_test_bal, y_pred_bal)
rec_bal = recall_score(y_test_bal, y_pred_bal)

comparison_text = (
    "TASK 5: BALANCED MODEL COMPARISON\n"
    f"New Accuracy:  {acc_bal:.4f}\n"
    f"New Precision: {prec_bal:.4f}\n"
    f"New Recall:    {rec_bal:.4f}\n"
)

print(comparison_text)
results_text.append(comparison_text)

with open("model_results.txt", "w") as f:
    f.writelines(results_text)

print("updatedddd'!")

TASK 5: BALANCED MODEL COMPARISON
New Accuracy:  1.0000
New Precision: 1.0000
New Recall:    1.0000

updatedddd'!
